## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image

from einops import rearrange

import numpy as np


## Model Architecture

### Processing Images

We will use a vision transformer pretrained using CLIP to extract features from images.

In [ ]:
from transformers import CLIPVisionModel

# d_model is 768
model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
model(pixel_values=torch.randn((1,3,224,224))).last_hidden_state.shape

### Processing Text

We will use a decoder only transformer to generate captions from the images.

#### Positional Encoding
Using rotary positional embeddings.

In [ ]:
class RoPE(nn.Module):
    def __init__(self, d_k: int):
        super().__init__()
        assert d_k % 2 == 0, "d_k must be even"
        self.d_k = d_k
        
        theta = 10000 ** (-2 * (torch.arange(0, d_k, 1) // 2) / d_k)
        self.register_buffer('theta', theta)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-1] == self.d_k

        # The batch size will be the first dimension
        # The sequence length will be the second to last dimension
        batch_size, seq_len = x.shape[0], x.shape[-2]
        
        # Create y by swapping pairs in the last dimension
        y = rearrange(x, "... (d j) -> ... d j", j=2)[..., [1, 0]]
        y = rearrange(y, "... d j -> ... (d j)")
        
        # Compute angles for each position and dimension
        positions = torch.arange(seq_len, dtype=torch.float32, device=x.device)
        thetas = positions.unsqueeze(-1) * self.theta.unsqueeze(0)  # (seq_len, d_model)
        
        # Compute cos and sin terms
        cos_terms = torch.cos(thetas)
        sin_terms = torch.sin(thetas)
        
        # Apply negation to sin terms for even indices
        sin_terms[:, 0::2] *= -1
        
        # Combine using rotation formula
        rotated_x = x * cos_terms.unsqueeze(0) + y * sin_terms.unsqueeze(0)
        return rotated_x

#### Multi Head Attention

In [ ]:

from typing import Optional

# creates an attention object, which computes the attention operation for the specific values of d_k
class Attention(nn.Module):
    def __init__(self, d_k: int, d_v: Optional[int] = None, decoder: bool = False):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v if d_v is not None else d_k
        self.decoder = decoder
        self.scale = np.sqrt(d_k)

    def forward(self,Q: torch.Tensor, K: torch.Tensor,V: torch.Tensor):
        '''
        Arguments:
            Q: Tensor of shape (batch_size, heads, seq_len, d_k)
            K: Tensor of shape (batch_size, heads, seq_len, d_k)
            V: Tensor of shape (batch_size, heads, seq_len, d_v)
        '''


        # the K tensor has its last 2 axes transposed
        x = torch.matmul(Q, torch.transpose(K,-2,-1)) / self.scale

        if self.decoder:
            seq_len = Q.shape[-2]

            # torch.full creates a matrix of a given size filled with a given element. torch.triu takes in a matrix and returns a matrix of the same size with all but elements above the main diagonal set to zero.
            # creates the attention mask that prevents future tokens from influencing past tokens
            mask = torch.triu(torch.full((seq_len, seq_len), -np.inf), diagonal=1).to(Q.device)

            x += mask

        # softmax over each row (a row, i, represents the "relevance" that each token, j, in the sequence has with token i)
        # creates the attention matrix
        x = F.softmax(x, -1)

        # for each token in position i, the new token in position i, after applying attention is the weighted sum of all tokens, j, using the weights from row i of the attention matrix
        x = torch.matmul(x, V)
        return x


# creates the multiheaded attention block for a transformer block
# can be customized to work for encoder or decoder blocks
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model: int, d_k: int, d_v: int, heads: int=8, decoder: bool = True):
        '''
        Arguments:
            d_model: int - the dimensionality of the model, or embeddings
            d_k: int - the dimensionality of vectors in each attention head
            d_v: int - the dimensionality of value vectors (should probably be the same as d_k)
            heads: int (default: 8) - the number of attention heads in the model
            decoder: bool (default: True) - set to True for decoder attention otherwise set to False
        '''

        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.h = heads
        self.decoder = decoder
        self.pe = RoPE(d_k)
        

        # can be decoder or encoder self attention
        self.attention = Attention(d_k, d_v, decoder=decoder)

        # input size should be the last dimension of the tensor, which is d_model
        self.query_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.key_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.value_weights = nn.Linear(in_features=d_model, out_features=d_v * self.h, bias=False)

        self.projection = nn.Linear(in_features=d_v * self.h, out_features=d_model)
    
    def forward(self, x: torch.Tensor):
        batch_size, seq_len, d_model = x.size() if len(x.size()) == 3 else (1,*x.size())

        # shape: (batch_size, seq_len, d_k * heads)
        queries = self.query_weights(x)
        keys = self.key_weights(x)

        # shape: (batch_size, seq_len, d_v * heads)
        values = self.value_weights(x)

        # using einops, rearrange the tensors to have shape (batch_size, heads, seq_len, d_k)
        queries = rearrange(queries, "batch seq_len (d_k heads) -> batch heads seq_len d_k", heads=self.h)
        keys = rearrange(keys, "batch seq_len (d_k heads) -> batch heads seq_len d_k", heads=self.h)
        values = rearrange(values, "batch seq_len (d_v heads) -> batch heads seq_len d_v", heads=self.h)
        
        # apply RoPE
        queries = self.pe(queries)
        keys = self.pe(keys)

        # size will be (batch_size, heads, seq_len, d_v)
        results = self.attention(queries, keys, values)

        results = rearrange(results, "batch heads seq_len d_v -> batch seq_len (d_v heads)")
        results = self.projection(results)
        return results



#### Cross Attention

In [ ]:
# creates the multiheaded attention block for a transformer block
# can be customized to work for encoder or decoder blocks
class CrossAttention(nn.Module):

    def __init__(self, d_model: int, d_k: int, d_v: int, heads: int=8):
        '''
        Arguments:
            d_model: int - the dimensionality of the model, or embeddings
            d_k: int - the dimensionality of vectors in each attention head
            d_v: int - the dimensionality of value vectors (should probably be the same as d_k)
            heads: int (default: 8) - the number of attention heads in the model
            decoder: bool (default: True) - set to True for decoder attention otherwise set to False
        '''

        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.h = heads

        # can be decoder or encoder self attention
        self.attention = Attention(d_k, d_v, decoder= False)

        # input size should be the last dimension of the tensor, which is d_model
        self.query_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.key_weights = nn.Linear(in_features=d_model, out_features=d_k * self.h, bias=False)
        self.value_weights = nn.Linear(in_features=d_model, out_features=d_v * self.h, bias=False)

        self.projection = nn.Linear(in_features=d_v * self.h, out_features=d_model)

    # 
    def forward(self, x: torch.Tensor, y: torch.tensor):
        '''
        Arguments:
            x: Tensor - The text embeddings
            y: Tensor - The image embeddings
        '''
        
        batch_size, seq_len, d_model = x.size() if len(x.size()) == 3 else (1,*x.size())
        batch_size, seq_len2, d_model = y.size() if len(y.size()) == 3 else (1,*y.size())

        # shape: (batch_size, seq_len, d_k * heads)
        queries = self.query_weights(x)

        # shape: (batch_size, seq_len2, d_k * heads)
        keys = self.key_weights(y)

        # shape: (batch_size, seq_len2, d_v * heads)
        values = self.value_weights(y)

        # using einops, rearrange the tensors to have shape (batch_size, heads, seq_len, d_k)
        queries = rearrange(queries, "batch seq_len (d_k heads) -> batch heads seq_len d_k", heads=self.h)
        keys = rearrange(keys, "batch seq_len2 (d_k heads) -> batch heads seq_len2 d_k", heads=self.h)
        values = rearrange(values, "batch seq_len2 (d_v heads) -> batch heads seq_len2 d_v", heads=self.h)

        # size will be (batch_size, heads, seq_len, d_v)
        results = self.attention(queries, keys, values)

        results = rearrange(results, "batch heads seq_len d_v -> batch seq_len (d_v heads)")
        results = self.projection(results)
        return results


### MLP Blocks
Following each multihead attention block is a MLP block.

In [ ]:

class MLP(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, activation: Optional[nn.Module]=None):
        super().__init__()
        self.activation = activation if activation is not None else F.gelu
        self.layer1 = nn.Linear(in_features=d_model, out_features=d_ff)
        self.layer2 = nn.Linear(in_features=d_ff, out_features=d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        x is a tensor of shape (batch_size, sequence_length, d_model)

        returns a tensor of shape (batch_size, sequence_length, d_model)
        '''

        return self.layer2(self.activation(self.layer1(x)))

### Transformer Blocks

We combine the multihead attention and cross attention with the MLP block with residuals and layer norms to create a single transformer block

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        d_model = 768,
        d_k = 64,
        heads = 12,
        d_ff = 3072,
        dropout = 0.1,
        *args,
        **kwargs
    ):
        '''
        Arguments:
            d_model: int (default: 768) - Representing the dimension of the model
            d_k: int (default: 64) - Representing the dimension of the key and value vectors in each head
            heads: int (default: 12) - Representing the number of heads in the multiheaded attention
            d_ff: int (default: 3072) - Representing the hidden dimension of the feedforward network
            dropout: float (default: 0.1) - Representing the dropout rate
            decoder: bool (default: True) - set to True for decoder attention otherwise set to False
        '''
        
        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.heads = heads
        self.d_ff = d_ff
        self.dropout = dropout

        self.layer_norm1 = nn.LayerNorm(normalized_shape=self.d_model)
        self.layer_norm2 = nn.LayerNorm(normalized_shape=self.d_model)
        self.layer_norm3 = nn.LayerNorm(normalized_shape=self.d_model)

        
        self.mha = MultiHeadAttention(
            d_model=self.d_model,
            d_k=self.d_k,
            d_v=self.d_k,
            heads=self.heads,
            decoder=True
        )

        self.ca = CrossAttention(
            d_model=self.d_model,
            d_k=self.d_k,
            d_v=self.d_k,
            heads=self.heads,
        )

        self.mlp = MLP(d_model=self.d_model, d_ff=self.d_ff)

        self.dropout1 = nn.Dropout(p=self.dropout)
        self.dropout2 = nn.Dropout(p=self.dropout)
        

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        '''
        x is a tensor of shape (batch_size, seq_len, d_model)

        y is a tensor of shape (batch_size, seq_len2, d_model)

        returns a tensor of shape (batch_size, seq_len, d_model)
        '''

        # Pre norm (deviates from original paper, but supposed to be better)
        residual = x
        x = self.layer_norm1(x)
        x = self.mha(x)
        x = self.dropout1(x) + residual

        residual = x
        x = self.layer_norm3(x)
        x = self.ca(x, y)
        x = self.dropout2(x) + residual

        residual = x
        x = self.layer_norm3(x)
        x = self.mlp(x)
        x = self.dropout2(x) + residual

        return x

### Putting an Image Captioner Together

We stack Transformer blocks to create a full Transformer. For the decoder language model, we use a linear head which is applied to the last token to predict the next token. We will use cross attention to let the text transformer to attend to the image embeddings.

In [ ]:

from transformers import CLIPVisionModel

class ImageCaptioner(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model = 768,
        d_k = 64,
        heads = 12,
        d_ff = 3072,
        dropout = 0.1,
        decoder = False,
        num_blocks = 10,
        max_seq_len = 512,
        **kwargs
    ):
        '''
        Arguments:
            d_model: int (default: 768) - Representing the dimension of the model
            d_k: int (default: 64) - Representing the dimension of the key and value vectors in each head
            heads: int (default: 12) - Representing the number of heads in the multiheaded attention
            d_ff: int (default: 3072) - Representing the hidden dimension of the feedforward network
            dropout: float (default: 0.1) - Representing the dropout rate
            decoder: bool (default: True) - set to True for decoder attention otherwise set to False
            num_blocks: int (default: 10) - Representing the number of encoder blocks
            vocab_size: int - The size of the vocabulary of the model
            max_seq_len: int - The maximum sequence length that the model can process
        '''
        super().__init__()

        self.d_model = d_model
        self.d_k = d_k
        self.heads = heads
        self.d_ff = d_ff
        self.dropout = dropout
        self.decoder = decoder
        self.num_blocks = num_blocks
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        
        self.lm_head = nn.Linear(in_features=self.d_model, out_features=self.vocab_size)
        
        # maps token ids to their embeddings
        self.embed = nn.Embedding(self.vocab_size, self.d_model)

        self.blocks = nn.ModuleList([TransformerBlock(
            d_model,
            d_k,
            heads,
            d_ff,
            dropout,
            decoder,
            max_seq_len
        ) for _ in range(self.num_blocks)])

        # d_model is also 768
        self.image_encoder = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        self.criterion = nn.CrossEntropyLoss(ignore_index=50257)

    def forward(self, image: torch.Tensor, tokens: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        '''
        x is a tensor of size (batch_size, seq_len) of token ids
        '''

        image_embeds = self.image_encoder(pixel_values = image).last_hidden_state
        x = self.embed(tokens)

        for block in self.blocks:
            x = block(x, image_embeds)
        
        x = self.lm_head(x)
        loss = self.criterion(x.view(-1, x.shape[-1]), labels.view(-1))
        
        return {
            "loss": loss
        }

    def generate(self, image: torch.Tensor, tokens: torch.Tensor, max_len: int = 100, temperature: float = 0.7) -> torch.Tensor:
        image_embeds = self.image_encoder(pixel_values = image).last_hidden_state
        

        for _ in range(max_len):
            x = self.embed(tokens)
            for block in self.blocks:
                x = block(x, image_embeds)
            x = self.lm_head(x[:, -1, :])

            x = x / temperature
            x = F.softmax(x, dim=-1)
            x = torch.multinomial(x, num_samples=1)

            tokens = torch.cat((tokens, x), dim=-1)

        return tokens


### Creating a Dataset

The dataset we will use is quite large, so we will stream it to prevent using up too much memory.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments
import torchvision.transforms as transforms
import torch
from torch.utils.data import IterableDataset
import wandb
import os

# Login to wandb
wandb.login(key=os.environ["WANDB_API_KEY"])

# Load dataset (streaming)
streaming_dataset = load_dataset("takara-ai/image_captions", split="train", streaming=True)

# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Image transformations
transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

# Dataset definition
class ImageCaptioningDataset(IterableDataset):
    def __init__(self, dataset, tokenizer, transform):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.transform = transform

    def __iter__(self):
        for item in self.dataset:
            image = self.transform(item["image"])
            caption = item["caption"]

            tokenized = self.tokenizer(
                caption, 
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=128
            )

            input_ids = tokenized.input_ids.squeeze()
            
            tokens = torch.cat((torch.tensor([50256]), input_ids[:-1]))

            yield {
                "image": image,
                "tokens": tokens,
                "labels": input_ids,
            }

ic_dataset = ImageCaptioningDataset(streaming_dataset, tokenizer, transform)


### Training loop

Here we train the model

In [ ]:

batch_size = 32
gpus = torch.cuda.device_count()
print(f"Number of GPUs: {gpus}")


# Custom collator function
def custom_collate_fn(batch):
    images = torch.stack([item["image"] for item in batch])
    tokens = torch.stack([item["tokens"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])

    return {
        "image": images,
        "tokens": tokens,
        "labels": labels,
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/",
    per_device_train_batch_size=(batch_size//gpus),
    evaluation_strategy="no",
    save_strategy="steps",
    save_steps=800,
    save_total_limit=3,
    logging_dir="./logs",
    logging_steps=50,
    max_steps=int(1000000 / batch_size),
    bf16=True,
    report_to="wandb",
    run_name="vlm-experiment",
)

model = ImageCaptioner(vocab_size=50258)

# Trainer initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ic_dataset,
    data_collator=custom_collate_fn
)

# Begin training
trainer.train()

### Loading the Model

In [ ]:
%pip install safetensors
import safetensors.torch as storch

In [ ]:

model = ImageCaptioner(vocab_size=50258)
model.load_state_dict(storch.load_file("better_data.safetensors"))
model.to("cuda")
model.eval()




In [ ]:
import torchvision.transforms as transforms

# Image transformations
transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])


# generate caption
image = transform(Image.open("tree.jpg"))
tokens = model.generate(image.unsqueeze(0).to("cuda"), torch.tensor([50256]).unsqueeze(0).to("cuda"), max_len=20, temperature=0.3)


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

print(tokenizer.decode(tokens[0].tolist()))
